# Predictive Analysis- Data Preprocessing
This notebook preprocesses our dataset and merges the 5 individual datasets into a master dataset, with target variables for 24-hour failure prediction (classification).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

### Loading all datasets

In [18]:
DATA_PATH = "drive/Othercomputers/My Mac/predictive-analysis/data/raw/"

telemetry_df = pd.read_csv(f'{DATA_PATH}PdM_telemetry.csv', parse_dates=['datetime'])
errors_df = pd.read_csv(f'{DATA_PATH}PdM_errors.csv', parse_dates=['datetime'])
maintenance_df = pd.read_csv(f'{DATA_PATH}PdM_maint.csv', parse_dates=['datetime'])
failures_df = pd.read_csv(f'{DATA_PATH}PdM_failures.csv', parse_dates=['datetime'])
machines_df = pd.read_csv(f'{DATA_PATH}PdM_machines.csv')

### Data filtering
Our dataset has maintainance data from 2014 but other data (like failure, telementry) starts from 2015. So, we need to filter our maintainace data to match other dataset and start from 2015.

In [19]:
print(f"Before: {len(maintenance_df):,} rows")
print(f"Date range: {maintenance_df['datetime'].min()} to {maintenance_df['datetime'].max()}")

maintenance_df = maintenance_df[maintenance_df['datetime'] >= '2015-01-01']

print(f"After: {len(maintenance_df):,} rows")
print(f"Date range: {maintenance_df['datetime'].min()} to {maintenance_df['datetime'].max()}")

Before: 3,286 rows
Date range: 2014-06-01 06:00:00 to 2016-01-01 06:00:00
After: 2,886 rows
Date range: 2015-01-01 06:00:00 to 2016-01-01 06:00:00


### Preparing events data

In this part, we are processing events data (errors, maintenance, failures) for each machine at every hour, and creating 3 pivot tables where each row represents the event status of a specific machine at a specific time. These tables include binary flags for each event type showing whether any event occurred during that time.
<br>This structure makes it easier to analyze a machine's condition over time: "when & what event happened to this machine?" and will be merged into master dataset later.


In [20]:
# Processing ERROR events
# binary flags for each error type
errors_pivot = errors_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='errorID',
    aggfunc='size',
    fill_value=0
).reset_index()

# renaming columns
errors_pivot.columns = ['machineID', 'datetime'] + [f'{col}' for col in errors_pivot.columns[2:]] # setting error type columns from pivoted table

# error flag
errors_pivot['has_error'] = (errors_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Error pivot columns: {list(errors_pivot.columns)}")


Error pivot columns: ['machineID', 'datetime', 'error1', 'error2', 'error3', 'error4', 'error5', 'has_error']


In [21]:
# Processing MAINTENANCE events
maintenance_pivot = maintenance_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='comp',
    aggfunc='size',
    fill_value=0
).reset_index()

# rename columns
maintenance_pivot.columns = ['machineID', 'datetime'] + [f'maint_{col}' for col in maintenance_pivot.columns[2:]]

# maintenance flag
maintenance_pivot['has_maintenance'] = (maintenance_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Maintenance columns: {list(maintenance_pivot.columns)}")


Maintenance columns: ['machineID', 'datetime', 'maint_comp1', 'maint_comp2', 'maint_comp3', 'maint_comp4', 'has_maintenance']


In [22]:
# Processing FAILURE events
failures_pivot = failures_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='failure',
    aggfunc='size',
    fill_value=0
).reset_index()

# rename columns
failures_pivot.columns = ['machineID', 'datetime'] + [f'failure_{col}' for col in failures_pivot.columns[2:]]

# failure flag
failures_pivot['has_failure'] = (failures_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Failure columns: {list(failures_pivot.columns)}")

Failure columns: ['machineID', 'datetime', 'failure_comp1', 'failure_comp2', 'failure_comp3', 'failure_comp4', 'has_failure']


### Merging dataset

Here, we are merging all our datasets into one master dataset. <br>
We take the telementry dataset as the base table by making a copy of it first, then do a left join on each of our pivoted events table one by one on the columns of machineID and datetime. Any row with a missing event data will be assinged a 'NaN' value on merge so we replace the NaN values with 0, representing the event as false for that entry.

In [23]:
# making a copy of telemetry data as the base
master_df = telemetry_df.copy()

# merging with machines dataset (no datetime on this)
master_df = master_df.merge(machines_df, on='machineID', how='left')

# merging with errors pivot dataset, replacing all NaN values with 0
master_df = master_df.merge(errors_pivot, on=['machineID', 'datetime'], how='left')
error_cols = [col for col in master_df.columns if col.startswith('error') or col == 'has_error']
master_df[error_cols] = master_df[error_cols].fillna(0).astype(int)

# merging with maintenance pivot dataset, replacing all NaN values with 0
master_df = master_df.merge(maintenance_pivot, on=['machineID', 'datetime'], how='left')
maint_cols = [col for col in master_df.columns if col.startswith('maint_') or col == 'has_maintenance']
master_df[maint_cols] = master_df[maint_cols].fillna(0).astype(int)

# mergeing with failures pivot dataset, replacing all NaN values with 0
master_df = master_df.merge(failures_pivot, on=['machineID', 'datetime'], how='left')
failure_cols = [col for col in master_df.columns if col.startswith('failure_') or col == 'has_failure']
master_df[failure_cols] = master_df[failure_cols].fillna(0).astype(int)

master_df = master_df.sort_values(['machineID', 'datetime']).reset_index(drop=True)

In [24]:
print("\nColumn summary of merged dataset:")
print(master_df.dtypes)


Column summary of merged dataset:
datetime           datetime64[ns]
machineID                   int64
volt                      float64
rotate                    float64
pressure                  float64
vibration                 float64
model                      object
age                         int64
error1                      int64
error2                      int64
error3                      int64
error4                      int64
error5                      int64
has_error                   int64
maint_comp1                 int64
maint_comp2                 int64
maint_comp3                 int64
maint_comp4                 int64
has_maintenance             int64
failure_comp1               int64
failure_comp2               int64
failure_comp3               int64
failure_comp4               int64
has_failure                 int64
dtype: object


### Creating 24hr Failure Prediction Target Variable

In [25]:
def create_failure_target(df, window = 24):
  """
  This creates binary target for failure prediction within specified window.

  For each row, it checks if there's a failure in the next window_hours and
  creates component-specific targets.
  """
  # creating a copy of the master_df
  df_copy = df.copy()

  # initiazing target columns
  df_copy['target_failure_24h'] = 0
  df_copy['target_comp1_24h'] = 0
  df_copy['target_comp2_24h'] = 0
  df_copy['target_comp3_24h'] = 0
  df_copy['target_comp4_24h'] = 0

  # the for loop basically works as:
  # For each machine:
    # For each failure event at time T:
      # Mark all rows from (T - 24h) to T as positive (1)
      # These row represent "failure will happen in next 24 hours"
  machines = df_copy['machineID'].unique() # process each machine separately
  for machine_id in machines:
    # filtering the master dataset for this machine's data
    machine_mask = df_copy['machineID'] == machine_id
    machine_data = df_copy[machine_mask].copy()

    # get failure times for this machine
    failure_times = machine_data[machine_data['has_failure'] == 1]['datetime'].values

    # if no failure, skip to next machine
    if len(failure_times) == 0:
      continue

    # for each row, check if any failure occurs within next window_hours
    for failure_time in failure_times:
      # window start = 24 hours (window) before time of failure
      start = failure_time - pd.Timedelta(hours=window)
      # windw end = time of failure
      end = failure_time

      # we mark all rows within the window as positive (failure will occur)
      time_window_mask = (
        (machine_data['datetime'] >= start) &
        (machine_data['datetime'] < end)
      )
      # get index for these rows at the master df copy
      window_indices = machine_data[time_window_mask].index

      # we set a new column 'target_failure_24h' with value = 1 on rows within the window
      # this marks as a general failure target
      df_copy.loc[window_indices, 'target_failure_24h'] = 1

      # to mark a component specific target: we check if the failure involved
      # that specific component and assign those rows with value = 1
      if machine_data.loc[machine_data['datetime'] == failure_time, 'failure_comp1'].values[0] == 1:
        df_copy.loc[window_indices, 'target_comp1_24h'] = 1
      if machine_data.loc[machine_data['datetime'] == failure_time, 'failure_comp2'].values[0] == 1:
        df_copy.loc[window_indices, 'target_comp2_24h'] = 1
      if machine_data.loc[machine_data['datetime'] == failure_time, 'failure_comp3'].values[0] == 1:
        df_copy.loc[window_indices, 'target_comp3_24h'] = 1
      if machine_data.loc[machine_data['datetime'] == failure_time, 'failure_comp4'].values[0] == 1:
        df_copy.loc[window_indices, 'target_comp4_24h'] = 1

  return df_copy

In [26]:
master_df = create_failure_target(master_df, 24)
print(f" Master dataset updated with {len(master_df)} rows.")

 Master dataset updated with 876100 rows.


In [30]:
print("\nTarget Variable Distribution:")
print("\nGeneral Failure (any component):")
target_counts = master_df['target_failure_24h'].value_counts()
print(f"No failure: {target_counts[0]:>8,} ({target_counts[0]/len(master_df)*100:.2f}%)")
print(f"Failure: {target_counts[1]:>8,} ({target_counts[1]/len(master_df)*100:.2f}%)")
print(f"Ratio: 1:{target_counts[0]//target_counts[1]}")

print("\nComponent-Specific Failures:")
for comp in ['comp1', 'comp2', 'comp3', 'comp4']:
    comp_col = f'target_{comp}_24h'
    comp_sum = master_df[comp_col].sum()
    print(f"{comp}: {comp_sum:>6,}")


Target Variable Distribution:

General Failure (any component):
No failure:  858,916 (98.04%)
Failure:   17,184 (1.96%)
Ratio: 1:49

Component-Specific Failures:
comp1:  4,581
comp2:  6,207
comp3:  3,135
comp4:  4,287
